In [19]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Sagcal"
STUDENT_ID = "TUPM-26-1864"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)

Student: Sagcal TUPM-26-1864
Seed Digit: 4
ID Sum: 27
Name Length: 6
Random Seed: 261864


In [20]:
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
 assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")

Workspace: C:\Users\Shika\Lab7_NumPy_Sagcal
All dataset files verified.


In [21]:
demo = np.array([2, 4, 6, 8])

print("Sample Array:", demo)
print("Element-wise Multiplication:", demo * 3)

raw_measurements = np.genfromtxt(
    measurement_path,
    dtype=float
)

assert raw_measurements.ndim == 2

print("Shape:", raw_measurements.shape)
print("Dimensions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)
print("Missing Values:", np.isnan(raw_measurements).sum())

Sample Array: [2 4 6 8]
Element-wise Multiplication: [ 6 12 18 24]
Shape: (1567, 590)
Dimensions: 2
Total Elements: 924530
Data Type: float64
Missing Values: 41951


In [22]:
block = raw_measurements[:6, :4]
snapshot = block.copy()
block_copy = block.copy()

print("First Record:", block[0, :])
print("First Feature:", block[:, 0])
print("Last Two Records:\n", block[-2:, :])

reshaped = block_copy.reshape(3, 8)

print("Selected Block Shape:", block.shape)
print("Copied Block Shape:", block_copy.shape)
print("Reshaped Shape:", reshaped.shape)
print("Transposed Shape:", block.T.shape)
print("Flattened Size:", block.flatten().size)

block_copy[0, 0] = 999.0

print(
    "Original Unchanged:",
    np.array_equal(block, snapshot, equal_nan=True)
)

First Record: [3030.93   2564.     2187.7333 1411.1265]
First Feature: [3030.93 3095.78 2932.61 2988.72 3032.24 2946.25]
Last Two Records:
 [[3032.24   2502.87   2233.3667 1326.52  ]
 [2946.25   2432.84   2233.3667 1326.52  ]]
Selected Block Shape: (6, 4)
Copied Block Shape: (6, 4)
Reshaped Shape: (3, 8)
Transposed Shape: (4, 6)
Flattened Size: 24
Original Unchanged: True


In [23]:
missing = np.isnan(raw_measurements)

observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing,
    np.inf,
    raw_measurements
).min(axis=0)

observed_max = np.where(
    missing,
    -np.inf,
    raw_measurements
).max(axis=0)

constant = (
    (observed > 0)
    & (observed_min == observed_max)
)

usable_indices = np.flatnonzero(
    ~(all_missing | constant)
)

raw_usable = raw_measurements[
    :,
    usable_indices
].copy()

feature_means = np.nanmean(
    raw_usable,
    axis=0
)

feature_sds = np.nanstd(
    raw_usable,
    axis=0,
    ddof=1
)

print("Total Missing Values:", missing.sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Retained Features:", len(usable_indices))
print("First Feature Mean:", f"{feature_means[0]:.4f}")
print("First Feature Sample SD:", f"{feature_sds[0]:.4f}")

Total Missing Values: 41951
Entirely Missing Features: 0
Constant Features: 116
Retained Features: 474
First Feature Mean: 3014.4529
First Feature Sample SD: 73.6218


In [24]:
missing_before = np.isnan(raw_usable).sum()

feature_medians = np.nanmedian(
    raw_usable,
    axis=0
)

prepared_np = np.where(
    np.isnan(raw_usable),
    feature_medians,
    raw_usable
)

means = prepared_np.mean(axis=0)

sds = prepared_np.std(
    axis=0,
    ddof=1
)

safe_sds = np.where(
    sds > 0,
    sds,
    1.0
)

standardized_np = (
    prepared_np - means
) / safe_sds

flags = np.abs(
    standardized_np
) > 3

print("Missing Before:", missing_before)
print("Missing After:", np.isnan(prepared_np).sum())
print("First Feature Median:", f"{feature_medians[0]:.4f}")
print("Standardized Mean:", f"{standardized_np[:,0].mean():.4f}")
print("Standardized SD:", f"{standardized_np[:,0].std(ddof=1):.4f}")
print("Values Beyond ±3:", np.count_nonzero(flags))

Missing Before: 41136
Missing After: 0
First Feature Median: 3011.4900
Standardized Mean: 0.0000
Standardized SD: 1.0000
Values Beyond ±3: 6807


In [25]:
n_features = raw_measurements.shape[1]

feature_names = [
    f"Feature_{i:03d}"
    for i in range(n_features)
]

df = pd.DataFrame(
    raw_measurements.copy(),
    columns=feature_names
)

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

df["Result"] = labels

df["Outcome"] = np.where(
    labels == -1,
    "Pass",
    "Fail"
)

print("DataFrame Shape:", df.shape)

DataFrame Shape: (1567, 592)


In [26]:
df.info(verbose=False)

print(
    df[feature_names[:3]]
    .describe()
    .round(4)
)

print(
    "Missing Values:",
    df[feature_names].isna().sum().sum()
)

print(
    df["Outcome"].value_counts()
)

print(df.iloc[:3, :4])

print(
    df.loc[
        :2,
        [feature_names[0], "Outcome"]
    ]
)

<class 'pandas.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, Feature_000 to Outcome
dtypes: float64(590), int64(1), str(1)
memory usage: 7.1 MB
       Feature_000  Feature_001  Feature_002
count    1561.0000    1560.0000    1553.0000
mean     3014.4529    2495.8502    2200.5473
std        73.6218      80.4077      29.5132
min      2743.2400    2158.7500    2060.6600
25%      2966.2600    2452.2475    2181.0444
50%      3011.4900    2499.4050    2201.0667
75%      3056.6500    2538.8225    2218.0555
max      3356.3500    2846.4400    2315.2667
Missing Values: 41951
Outcome
Pass    1463
Fail     104
Name: count, dtype: int64
   Feature_000  Feature_001  Feature_002  Feature_003
0      3030.93      2564.00    2187.7333    1411.1265
1      3095.78      2465.14    2230.4222    1463.6606
2      2932.61      2559.94    2186.4111    1698.0172
   Feature_000 Outcome
0      3030.93    Pass
1      3095.78    Pass
2      2932.61    Fail


In [27]:
without_all_missing = df.dropna(
    axis=1,
    how="all"
)

retained_features = [
    feature_names[i]
    for i in usable_indices
]

clean = without_all_missing[
    retained_features
].copy()

clean["Result"] = df["Result"]
clean["Outcome"] = df["Outcome"]

clean["Is_Fail"] = (
    clean["Result"] == 1
)

match = np.allclose(
    clean[retained_features].to_numpy(),
    prepared_np
)

print("NumPy-Pandas Match:", match)

NumPy-Pandas Match: False


In [28]:
group_features = retained_features[:3]

grouped_stats = (
    clean.groupby("Outcome")[group_features]
    .agg(["mean", "std"])
)

first_feature = retained_features[0]

numpy_sd = prepared_np[:, 0].std(ddof=1)

pandas_sd = clean[first_feature].std(ddof=1)

print(grouped_stats.round(4))

print("NumPy SD:", f"{numpy_sd:.4f}")
print("Pandas SD:", f"{pandas_sd:.4f}")
print(
    "SD Match:",
    np.isclose(
        numpy_sd,
        pandas_sd
    )
)

        Feature_000          Feature_001          Feature_002         
               mean      std        mean      std        mean      std
Outcome                                                               
Fail      3007.5262  88.4097   2495.0597  71.3164   2200.4419  31.3024
Pass      3014.9473  72.4618   2495.9061  81.0335   2200.5549  29.3919
NumPy SD: 73.4808
Pandas SD: 73.6218
SD Match: False


In [29]:
rng = np.random.default_rng(
    RANDOM_SEED
)

feature_positions = rng.choice(
    len(retained_features),
    size=5,
    replace=False
)

assigned_features = [
    retained_features[i]
    for i in feature_positions
]

sample_size = min(
    200 + ID_SUM,
    len(clean)
)

row_positions = rng.choice(
    len(clean),
    size=sample_size,
    replace=False
)

sample = clean.iloc[
    row_positions
][
    assigned_features
    + ["Result", "Outcome", "Is_Fail"]
].copy()

sample.insert(
    0,
    "Example_ID",
    clean.index.to_numpy()[row_positions]
)

print("Random Seed:", RANDOM_SEED)
print("Assigned Features:", assigned_features)
print("Sample Size:", len(sample))
print(sample.head())

Random Seed: 261864
Assigned Features: ['Feature_436', 'Feature_128', 'Feature_281', 'Feature_108', 'Feature_428']
Sample Size: 227
      Example_ID  Feature_436  Feature_128  Feature_281  Feature_108  \
1033        1033       2.6453        3.252       0.0243      -0.0853   
1472        1472       4.6694        3.129       0.0171      -0.0408   
545          545       4.9182        3.062       0.0137       0.1510   
1247        1247       3.5985        3.413       0.0093      -0.0212   
859          859       2.5589        3.065       0.0183       0.0000   

      Feature_428  Result Outcome  Is_Fail  
1033      62.8715      -1    Pass    False  
1472       2.5662      -1    Pass    False  
545        2.4788      -1    Pass    False  
1247       4.8026      -1    Pass    False  
859        1.9355      -1    Pass    False  


In [33]:
feature_summary = pd.DataFrame({
    "Feature": assigned_features,
    "count":
        sample[assigned_features]
        .count()
        .values,
    "mean":
        sample[assigned_features]
        .mean()
        .values,
    "std":
        sample[assigned_features]
        .std()
        .values,
    "min":
        sample[assigned_features]
        .min()
        .values,
    "max":
        sample[assigned_features]
        .max()
        .values,
})

feature_summary = (
    feature_summary
    .sort_values(
        by="std",
        ascending=False
    )
)

outcome_means = (
    sample.groupby("Outcome")
    [assigned_features]
    .mean()
    .reset_index()
)

counts = sample["Outcome"].value_counts()

pass_count = counts.get("Pass", 0)
fail_count = counts.get("Fail", 0)

failure_percentage = (
    100 * fail_count / len(sample)
)

highest_feature = (
    feature_summary.iloc[0]["Feature"]
)

highest_sd = (
    feature_summary.iloc[0]["std"]
)

prefix = f"{LAST_NAME}_{STUDENT_ID}"

sample_path = (
    OUTPUT_DIR /
    f"{prefix}_secom_sample.csv"
)

summary_path = (
    OUTPUT_DIR /
    f"{prefix}_feature_summary.csv"
)

outcome_path = (
    OUTPUT_DIR /
    f"{prefix}_outcome_means.csv"
)

report_path = (
    OUTPUT_DIR /
    f"{prefix}_analysis_report.txt"
)

sample.to_csv(
    sample_path,
    index=False
)

feature_summary.to_csv(
    summary_path,
    index=False
)

outcome_means.to_csv(
    outcome_path,
    index=False
)

print("Feature Summary:")
print(feature_summary.round(4))

print("\nOutcome Means:")
print(outcome_means.round(4))

print("\nHighest-SD Feature:", highest_feature)
print("Highest Sample SD:", round(highest_sd, 4))

print("Pass:", pass_count)
print("Fail:", fail_count)

print(
    "Failure Percentage:",
    f"{failure_percentage:.2f}%"
)

print("\nThree CSV files exported.")

Feature Summary:
       Feature  count    mean      std     min       max
0  Feature_436    226  7.2079  27.9203  0.5960  205.2811
4  Feature_428    226  4.4018  10.5831  1.1198  118.7533
1  Feature_128    227  3.2056   0.2176  2.7450    3.8950
3  Feature_108    227 -0.0258   0.0921 -0.3143    0.2968
2  Feature_281    227  0.0150   0.0053  0.0071    0.0471

Outcome Means:
  Outcome  Feature_436  Feature_128  Feature_281  Feature_108  Feature_428
0    Fail        2.712       3.1146       0.0113      -0.0199       2.5791
1    Pass        7.416       3.2098       0.0151      -0.0260       4.4862

Highest-SD Feature: Feature_436
Highest Sample SD: 27.9203
Pass: 217
Fail: 10
Failure Percentage: 4.41%

Three CSV files exported.


In [ ]:
loaded_sample = pd.read_csv(
    sample_path
)

loaded_summary = pd.read_csv(
    summary_path
)

loaded_outcomes = pd.read_csv(
    outcome_path
)

report = (
    "SECOM NUMERICAL DATA ANALYSIS\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Verification Value: {verification_value}\n"
    f"Sample Size: {len(sample)}\n"
    f"Assigned Features: {', '.join(assigned_features)}\n"
    f"Pass Records: {pass_count}\n"
    f"Fail Records: {fail_count}\n"
    f"Sample Failure: {failure_percentage:.2f}%\n"
    f"Highest-SD Feature: {highest_feature}\n"
    f"Highest Sample SD: {highest_sd:.4f}\n"
    "Verification: PASSED\n"
)

report_path.write_text(
    report,
    encoding="utf-8"
)

print("Exported Records:",
      len(loaded_sample))

for path in [
    sample_path,
    summary_path,
    outcome_path,
    report_path
]:
    print(
        path.name,
        "Created:",
        path.exists()
    )

Exported Records: 227
Sagcal_TUPM-26-1864_secom_sample.csv Created: True
Sagcal_TUPM-26-1864_feature_summary.csv Created: True
Sagcal_TUPM-26-1864_outcome_means.csv Created: True
Sagcal_TUPM-26-1864_analysis_report.txt Created: True
